### imports the web requests package, establishes your working directory path, and defines the exact data codes used by the World Bank for Singapore.

In [11]:
import os
import wbgapi as wb
import pandas as pd

# Define data storage path
RAW_DATA_PATH = "../data/raw"
os.makedirs(RAW_DATA_PATH, exist_ok=True)

# Set country code to Singapore (ISO3 format)
COUNTRY_CODE = "SGP"

# Map official World Bank indicators to clean names
INDICATORS = {
    "NY.GDP.MKTP.CD": "gdp_nominal_usd",
    "NY.GDP.MKTP.KD.ZG": "gdp_growth_annual_pct",
    "FP.CPI.TOTL.ZG": "inflation_rate_pct",
    "NE.EXP.GNFS.ZS": "exports_pct_gdp",
    "GC.XPN.TOTL.GD.ZS": "gov_expense_pct_gdp",
    "SP.POP.TOTL": "total_population",
    "SP.POP.1564.TO.ZS": "working_age_pop_pct",
    "SE.XPD.TOTL.GD.ZS": "education_expenditure_pct_gdp"
}
print("[READY] Configuration set for Singapore.")


[READY] Configuration set for Singapore.


The World Bank API wraps its data inside a nested JSON structure where the first item contains metadata (like pagination details) and the second item contains the actual timeline records. This function clean-parses that logic safely.

#### execute your logic. It will download the data directly into your local workspace.

In [12]:
print(f"Connecting to World Bank servers for {COUNTRY_CODE}...")

try:
    # Fetching all indicators using the official wrapper package
    raw_df = wb.data.DataFrame(
        series=list(INDICATORS.keys()), 
        economy=COUNTRY_CODE, 
        time=range(1998, 2026),
        numericTimeKeys=True
    )
    
    # Flip the table structure so years become rows
    raw_df = raw_df.T
    raw_df.index.name = 'year'
    raw_df = raw_df.reset_index()
    
    # Change alphanumeric codes to clean english headers
    raw_df = raw_df.rename(columns=INDICATORS)
    
    # Save file to disk if rows were successfully returned
    if len(raw_df) > 0:
        output_file = os.path.join(RAW_DATA_PATH, "singapore_worldbank_raw.csv")
        raw_df.to_csv(output_file, index=False)
        print(f"\n[SUCCESS] Successfully wrote {len(raw_df)} historical rows to file!")
        print(raw_df.head(5))
    else:
        print("\n[ERROR] Server connected, but table returned zero data rows.")

except Exception as e:
    print(f"\n[CRITICAL ERROR] Download failed: {e}")


Connecting to World Bank servers for SGP...

[SUCCESS] Successfully wrote 28 historical rows to file!
series  year  inflation_rate_pct  gov_expense_pct_gdp  exports_pct_gdp  \
0       1998           -0.267502            15.721914       166.787571   
1       1999            0.016710            14.772397       176.745013   
2       2000            1.361624            15.716732       188.350890   
3       2001            0.997198            18.169432       182.894181   
4       2002           -0.391677            16.145629       184.086118   

series  gdp_nominal_usd  gdp_growth_annual_pct  education_expenditure_pct_gdp  \
0          8.572821e+10              -2.191015                            NaN   
1          8.628685e+10               5.718372                            NaN   
2          9.607654e+10               9.038316                        3.32129   
3          8.979379e+10              -1.070863                        3.54250   
4          9.253837e+10               3.923361  

## Argentina - com

In [13]:
# --- ADDING ARGENTINA PROFILE DATA ---
COUNTRY_CODE_ARG = "ARG"

print(f"Connecting to World Bank servers for {COUNTRY_CODE_ARG}...")

try:
    # Fetching all matching structural indicators for Argentina
    arg_raw_df = wb.data.DataFrame(
        series=list(INDICATORS.keys()), 
        economy=COUNTRY_CODE_ARG, 
        time=range(1998, 2026),
        numericTimeKeys=True
    )
    
    # Reshape table formatting
    arg_raw_df = arg_raw_df.T
    arg_raw_df.index.name = 'year'
    arg_raw_df = arg_raw_df.reset_index().rename(columns=INDICATORS)
    
    # Export raw asset to disk
    if len(arg_raw_df) > 0:
        arg_output_file = os.path.join(RAW_DATA_PATH, "argentina_worldbank_raw.csv")
        arg_raw_df.to_csv(arg_output_file, index=False)
        print(f"\n[SUCCESS] Successfully wrote {len(arg_raw_df)} historical rows for Argentina!")
        print(arg_raw_df.head(3))
        
except Exception as e:
    print(f"\n[CRITICAL ERROR] Download failed for Argentina: {e}")


Connecting to World Bank servers for ARG...

[SUCCESS] Successfully wrote 28 historical rows for Argentina!
series  year  inflation_rate_pct  gov_expense_pct_gdp  exports_pct_gdp  \
0       1998                 NaN            15.098801        10.415582   
1       1999                 NaN            16.743686         9.827175   
2       2000                 NaN            16.832888        10.986375   

series  gdp_nominal_usd  gdp_growth_annual_pct  education_expenditure_pct_gdp  \
0          2.989482e+11               3.850179                        4.03987   
1          2.835230e+11              -3.385457                        4.52168   
2          2.842038e+11              -0.788999                        4.58031   

series  working_age_pop_pct  total_population  
0                 61.871353        36372860.0  
1                 62.025624        36794682.0  
2                 62.173611        37213984.0  


##  Argentina's Economic Health Score

In [14]:
# --- SELF-CONTAINED PROCESSING ENGINE FOR ARGENTINA ---
import os
import pandas as pd
import numpy as np

# Re-define paths locally to ensure the cell runs independently
RAW_DATA_PATH = "../data/raw"
PROCESSED_DATA_PATH = "../data/processed"
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

arg_raw_file = os.path.join(RAW_DATA_PATH, "argentina_worldbank_raw.csv")
arg_processed_file = os.path.join(PROCESSED_DATA_PATH, "argentina_processed_matrix.csv")

# Re-define normalization function locally to prevent NameErrors
def min_max_scale_local(series):
    return (series - series.min()) / (series.max() - series.min()) if (series.max() - series.min()) != 0 else 0

if os.path.exists(arg_raw_file):
    df_arg = pd.read_csv(arg_raw_file).sort_values("year").reset_index(drop=True)
    
    # Run data imputation pipeline
    df_arg_clean = df_arg.ffill().bfill()
    
    # Min-Max normalization scaling
    df_arg_clean['scaled_growth'] = min_max_scale_local(df_arg_clean['gdp_growth_annual_pct'])
    df_arg_clean['scaled_exports'] = min_max_scale_local(df_arg_clean['exports_pct_gdp'])
    df_arg_clean['scaled_inflation'] = 1 - min_max_scale_local(df_arg_clean['inflation_rate_pct'])
    
    # Generate weighted Economic Health Score
    df_arg_clean['economic_health_score'] = (
        (df_arg_clean['scaled_growth'] * 0.40) + 
        (df_arg_clean['scaled_exports'] * 0.40) + 
        (df_arg_clean['scaled_inflation'] * 0.20)
    ) * 100
    
    # Export clean analytical matrix to disk
    df_arg_clean.to_csv(arg_processed_file, index=False)
    print("[SUCCESS] Argentina data matrix processed, calculated, and saved successfully!")
    print(df_arg_clean[['year', 'inflation_rate_pct', 'economic_health_score']].tail(3))
else:
    print(f"[ERROR] Raw Argentina file not found at: {arg_raw_file}")


[SUCCESS] Argentina data matrix processed, calculated, and saved successfully!
    year  inflation_rate_pct  economic_health_score
25  2023          133.488936              32.711302
26  2024          219.883929              29.596142
27  2025          219.883929              29.596142
